# ChoiEtAl2021-JCAM-IGQuad

### Title:
__Inverse Gaussian Quadrature and Finite Normal-Mixture Approximation of the Generalized Hyperbolic Distribution__

### Authors:
* Jaehyuk Choi
* Younhee Du
* Qing Song

### Journal:
Journal of Computational and Applied Mathematics 388 (2021) 113302.  
https://doi.org/10.1016/j.cam.2020.113302

### Abstract:
This study presents novel numerical quadratures for the inverse Gaussian (IG) and generalized inverse Gaussian (GIG) distributions,
derived from the Gaussâ€“Hermite quadrature.
The quadrature is applied to efficiently approximate the CDF of the generalized hyperbolic (GH) distribution
as a finite normal mixture.
The method achieves exponential convergence and is orders of magnitude faster than adaptive density integration.
It also yields an efficient GIG random-variate generator for Monte Carlo simulation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as spst
import scipy.special as spsp
import scipy.optimize as spop

In [ ]:
### if you want to run on your modified PyFeng code.
#%load_ext autoreload
#%autoreload 2

In [ ]:
# Install pyfeng if not available (e.g., in Google Colab)
try:
    ### Uncomment below if you want to run on your modified code
    #import sys
    #sys.path.insert(sys.path.index('')+1, 'YOUR_LOCAL_PyFeng_PATH')
    import pyfeng as pf
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "pyfeng"])
    import pyfeng as pf

## GH Distribution Setup

The GH distribution is a normal variance-mean mixture:
$$X = \mu + \beta W + \sqrt{W}\,Z, \quad Z\sim N(0,1), \quad W\sim\text{GIG}(\gamma,\delta,p)$$
where $\gamma = \sqrt{\alpha^2-\beta^2}$.

The CDF is approximated via GIG quadrature (Eq. 3):
$$F_{\text{GH}}(y) \approx \sum_{k=1}^{n} w_k\,\Phi\!\left(\frac{y-\mu-\beta x_k}{\sqrt{x_k}}\right)$$
where $(x_k, w_k)$ are the GIG quadrature nodes and weights from `pf.DistGig.quad()`.

Derived parameters: $\sigma = \sqrt{\gamma\delta}$ (concentration), $\tilde{\beta} = \beta\sqrt{\delta/\gamma}$ (normalised asymmetry).

In [ ]:
# Parameter sets from Table 1: (mu, beta, gamma, delta, p)
# gamma = sqrt(alpha^2 - beta^2); alpha is given in the paper
param_sets = [
    pf.DistGh(mu=0,        beta=0,        gamma=np.sqrt(1**2        - 0**2       ), delta=1,      p=-0.5  ),
    pf.DistGh(mu=0.00029,  beta=-4.90461, gamma=np.sqrt(138.78464**2-4.90461**2  ), delta=0.00646, p=-0.5  ),
    pf.DistGh(mu=0.000666, beta=-6.17,    gamma=np.sqrt(214.4**2    - 6.17**2    ), delta=0.0022,  p= 0.8357),
    pf.DistGh(mu=0.000048, beta=2.73,     gamma=np.sqrt(9**2        - 2.73**2    ), delta=0.0161,  p=-1.663 ),
]
labels = ['Set 1', 'Set 2', 'Set 3', 'Set 4']

def gh_percentile_grid(gh):
    """99 equi-spaced percentiles using the current quadrature stored in gh."""
    qs = np.arange(1, 100) / 100
    return np.array([gh.ppf(q) for q in qs]), qs


## Table 1 â€” GH Parameter Sets and Statistical Moments

In [ ]:
rows = []
for gh, label in zip(param_sets, labels):
    m1, mc2, skew, exkurt = gh.mvsk()
    rows.append({
        '': label,
        'mu': gh.mu,    'alpha': round(gh.alpha, 5), 'beta': gh.beta,
        'delta': gh.delta, 'p': gh.p,
        'sigma': round(gh.sigma, 4), 'beta_tilde': round(gh.beta_tilde, 4),
        'mean':     f'{m1:.2E}',
        'variance': f'{mc2:.2E}',
        'skewness': round(skew, 3),
        'ex-kurtosis': round(exkurt, 3),
    })
pd.DataFrame(rows).set_index('')

## Figure 2 â€” MGF Convergence of GIG Quadrature

For $W\sim\text{GIG}(\sigma,\sigma,p)$, the exact MGF at $t=0.4\sigma^2$ (80% of convergence radius $\sigma^2/2$) is:
$$M_W(t) = \left(\frac{\gamma^2}{\gamma^2-2t}\right)^{p/2}\frac{K_p\bigl(\delta\sqrt{\gamma^2-2t}\bigr)}{K_p(\delta\gamma)}$$
evaluated stably via `kve` (exponentially scaled Bessel).
The quadrature approximation is $\hat{M}_W(t)=\sum_k w_k e^{t x_k}$.

In [ ]:
def gig_mgf_exact(t, gamma, delta, p):
    """Exact GIG MGF using kve for numerical stability."""
    g_new = np.sqrt(gamma**2 - 2*t)
    eta, eta_new = delta*gamma, delta*g_new
    # K_p(x) = kve(p,x)*exp(-x)  =>  ratio uses kve and exp(eta-eta_new)
    return (gamma/g_new)**p * spsp.kve(p, eta_new) / spsp.kve(p, eta) * np.exp(eta - eta_new)

def gig_mgf_quad(t, gamma, delta, p, n):
    x, w = pf.DistGig(gamma=gamma, delta=delta, p=p).quad(n)
    return w @ np.exp(t * x)

p_vals   = [-0.5,             1,              90]
p_titles = ['p = -0.5 (NIG)', 'p = 1 (hyperbolic)', 'p = 90']
sigma_vals = [0.5, 1.0, 1.5, 2.0]
n_arr = np.arange(5, 61)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
markers = ['o', 's', '^', 'x']
colors  = ['b', 'r', 'g', 'k']

for ax, p, title in zip(axes, p_vals, p_titles):
    for sigma, mk, col in zip(sigma_vals, markers, colors):
        t = 0.4 * sigma**2
        mgf_ref = gig_mgf_exact(t, sigma, sigma, p)
        errs = [abs(gig_mgf_quad(t, sigma, sigma, p, n) - mgf_ref) for n in n_arr]
        ax.semilogy(n_arr, errs, marker=mk, color=col, label=f'sigma={sigma}', markersize=3, linewidth=0.8)
    ax.set_xlabel('n (quadrature size)', fontsize=11)
    ax.set_ylabel('|Error|', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(linestyle='--', alpha=0.6)

plt.suptitle('Figure 2: MGF Convergence of GIG Quadrature', fontsize=13)
plt.tight_layout()
plt.show()

## Figure 3 â€” Convergence of GH CDF

Maximum absolute error over 99 equi-spaced percentiles $\{y_j = F^{-1}(j/100): j=1,\ldots,99\}$
as a function of quadrature size $n$, for the four GH parameter sets in Table 1.

In [ ]:
n_arr_cdf = np.arange(20, 101, 5)

# Use a high-accuracy reference (n=500) to compute the percentile grid once per set
fig, ax = plt.subplots(figsize=(7, 5))
markers_ps = ['o', '^', '+', 'x']
colors_ps  = ['b', 'r', 'g', 'k']

for gh, label, mk, col in zip(param_sets, labels, markers_ps, colors_ps):
    gh.n_quad = 500
    y_q, qs = gh_percentile_grid(gh)           # reference at n=500

    errs = []
    for n in n_arr_cdf:
        gh.n_quad = n
        errs.append(np.max(np.abs(gh.cdf(y_q) - qs)))
    ax.semilogy(n_arr_cdf, errs, marker=mk, color=col,
                label=label, markersize=5, linewidth=1)

ax.set_xlabel('n (quadrature size)', fontsize=12)
ax.set_ylabel('Max |Error| over 99 percentiles', fontsize=11)
ax.set_title('Figure 3: GH CDF Convergence', fontsize=13)
ax.legend(fontsize=10)
ax.grid(linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Figure 4 â€” CDF Error as Functions of Parameters

For $Y\sim\text{GH}(0,\tilde{\beta},\sigma,\sigma,p)$, the base case is Set 1: $\tilde{\beta}=0,\,\sigma=1,\,p=-0.5$.
Each panel varies one parameter while the others are held at their base values.

In [ ]:
n_test_vals = [60, 80, 100]
ls_styles   = ['-', '--', ':']
col_styles  = ['b', 'r', 'g']

def max_cdf_err(gh, n_test, n_ref=200):
    gh.n_quad = n_ref;  y_q, qs = gh_percentile_grid(gh)
    gh.n_quad = n_test; return np.max(np.abs(gh.cdf(y_q) - qs))

fig, axes = plt.subplots(3, 1, figsize=(7, 10))

# Upper: vary |beta_tilde| (gamma=delta=1, p=-0.5  =>  beta=beta_tilde)
bt_arr = np.arange(0, 4.01, 0.2)
ax = axes[0]
errs_bt = {n_t: [] for n_t in n_test_vals}
for bt in bt_arr:
    gh = pf.DistGh(mu=0., beta=bt, gamma=1., delta=1., p=-0.5)
    for n_t in n_test_vals:
        errs_bt[n_t].append(max_cdf_err(gh, n_t))
for n_t, ls, col in zip(n_test_vals, ls_styles, col_styles):
    ax.semilogy(bt_arr, errs_bt[n_t], ls=ls, color=col, label=f'n={n_t}', linewidth=1.5)
ax.set_xlabel('|beta_tilde|', fontsize=12);  ax.set_ylabel('Max |Error|', fontsize=11)
ax.legend(fontsize=10);  ax.grid(linestyle='--', alpha=0.6)

# Middle: vary sigma (gamma=delta=sigma, beta=0, p=-0.5)
sigma_arr = np.arange(0.1, 1.01, 0.05)
ax = axes[1]
errs_sg = {n_t: [] for n_t in n_test_vals}
for sigma in sigma_arr:
    gh = pf.DistGh(mu=0., beta=0., gamma=float(sigma), delta=float(sigma), p=-0.5)
    for n_t in n_test_vals:
        errs_sg[n_t].append(max_cdf_err(gh, n_t))
for n_t, ls, col in zip(n_test_vals, ls_styles, col_styles):
    ax.semilogy(sigma_arr, errs_sg[n_t], ls=ls, color=col, label=f'n={n_t}', linewidth=1.5)
ax.set_xlabel('sigma', fontsize=12);  ax.set_ylabel('Max |Error|', fontsize=11)
ax.legend(fontsize=10);  ax.grid(linestyle='--', alpha=0.6)

# Lower: vary p (gamma=delta=1, beta=0)
p_arr = np.arange(-15, 15.1, 0.5);  p_arr = p_arr[p_arr != 0]
ax = axes[2]
errs_p = {n_t: [] for n_t in n_test_vals}
for p_val in p_arr:
    gh = pf.DistGh(mu=0., beta=0., gamma=1., delta=1., p=float(p_val))
    for n_t in n_test_vals:
        errs_p[n_t].append(max_cdf_err(gh, n_t))
for n_t, ls, col in zip(n_test_vals, ls_styles, col_styles):
    ax.semilogy(p_arr, errs_p[n_t], ls=ls, color=col, label=f'n={n_t}', linewidth=1.5)
ax.set_xlabel('p', fontsize=12);  ax.set_ylabel('Max |Error|', fontsize=11)
ax.legend(fontsize=10);  ax.grid(linestyle='--', alpha=0.6)

plt.suptitle('Figure 4: CDF Error vs Parameters', fontsize=13)
plt.tight_layout()
plt.show()

## Table 3 â€” CDF Error at Extreme Quantiles ($n = 50$)

Error $= F_{\text{quad}}(y_q) - q$, where $y_q = F^{-1}(q)$ is the reference quantile (computed with $n=500$).

In [ ]:
n_test  = 50
q_vals  = [1e-9, 1e-6, 1e-3, 1-1e-3, 1-1e-6, 1-1e-9]
q_labels= ['1e-9', '1e-6', '1e-3', '1-1e-3', '1-1e-6', '1-1e-9']

results = {}
for gh, label in zip(param_sets, labels):
    gh.n_quad = 500
    y_refs = [gh.ppf(q) for q in q_vals]       # reference quantiles at n=500

    gh.n_quad = n_test
    results[label] = [gh.cdf(y_q) - q for y_q, q in zip(y_refs, q_vals)]

df_t3 = pd.DataFrame(results, index=q_labels)
df_t3.apply(lambda c: c.map(lambda v: f'{v:.2E}'))